# Implementing a simple Retrieval-augmented generation (RAG)
This Lab is based on a workshop created by our friends of the [Swiss-AI-Center](https://www.hes-so.ch/swiss-ai-center).

Original Authors:
- Jonathan Guerne, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Henrique Marques Reis, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Célien Donzé, Scientific Collaborator at HEIA-FR, Switzerland

Adapted by:
- Francesco Carrino, Professor at Haute Ecole Ingénierie, Valais/Wallis Switzerland

## Introduction
This notebook explains how to create a RAG (Retrieval Augmented Generation) system to answer questions about online or PDF documents. We will use a self-hosted LLM with Ollama to generate answers to the questions. Additionally, we will use a vector database to retrieve relevant documents for answering the questions.

### Documentation and references
Technologies and tools used in this lab:
- [Large Language Model (LLM)](https://en.wikipedia.org/wiki/Large_language_model), a model that generates text (typically) baseded on the transformer architecture. In this lab, we will used opensource models like LLAMA 3.x.
- [Ollama](https://ollama.com/): Ollama is a platform that democratizes access to LLMs by enabling users to run them locally on their machines. In this lab, it will be used to run the LLMs on your machine or in the cloud (Kaggle).
- [Vector database](https://en.wikipedia.org/wiki/Vector_database): A Vector Database is a relational database system specifically designed to process vectorized data. Unlike conventional databases that contain information in tables, rows, and columns, vector databases work with vectors–arrays of numerical values that signify points in multidimensional space. When used with text data, vector databases can store and retrieve information based on the semantic meaning of the text, rather than just the text itself. Through a process called embedding, text data is converted into vectors that represent the **meaning** of the text. This meanse text with similar meaning is stored close to each other in the vector space. We will use [FAISS](https://python.langchain.com/docs/integrations/vectorstores/faiss/) an opensource library for efficient similarity search and clustering of dense vectors. In this lab, it will be used to store information about pdf and web documents.
- [LangChain](https://python.langchain.com/docs/introduction/): LangChain is a framework for developing applications powered by large language models (LLMs).  It is based on the idea of "chains", where a chain defines sequences of actions, where each step can involve querying an LLM, prompt management, manipulating data, or interacting with external tools applications. In this lab, it will be used to put together the LLM and the vector database to create a RAG system.
- [Retrieval-augmented generation (RAG)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation): RAG is a technique to improve LLMs by adding a retrieval component from an external data source. This is the main goal of this tutorial.

### Structure and goals
This lab is divided into 4 parts. The first 3 parts, you will be guided to install and implement a simple RAG working on pdf files. Your goal in this part understand the code. We provide some questions to guide your learning through the code.

The last part is a challenge where you will have to use what you learnt to implement a RAG working on web pages.

1. **Installation**: Install the necessary libraries and download the data. We provide two guides: one for running the code on Kaggle (or Google Colab), and another for running the code locally.
2. **Creating and manipulating prompts**: We will create prompts and create simple query the LLM and the vector database.
3. **Creating a RAG system**: We will create a simple RAG system to answer questions about pdf documents. The pdf will be embedded in the vector database and the LLM will generate the answers.
4. **Challenge**: In this last part, *you* will implement a RAG system to answer questions about web pages.

---

## 1. Installation



If your machine is not very performant, you can use Kaggle or Google Colab to run this notebook. The following code block installs the necessary packages.
If you choose to run the code on Kaggle, login to your account and import this notebook. It is raccomanded to have a verified account to have access to the GPU.

Below you can find the instraction to install the necessary packages either on Kaggle (Section 1.1) or on your local machine (Section 1.2).

### 1.1 Installing Ollama (Kaggle or Google Colab)

If you are in Kaggle or Google Colab, you can install Ollama using the following code block. If want to run this notebook locally, see the next section.

These instructions are based on the Medium post titled [How to Run Ollama Models on Google Colab and Kaggle: A Complete Guide](https://medium.com/@mradul18varshney/how-to-run-ollama-generative-ai-models-on-google-colab-and-kaggle-a-complete-guide-3e01fc12512f).

In [ ]:
# Package installation (ONLY on Kaggle or Google Colab)
!pip install langchain langchain_huggingface langchain-community faiss-cpu pymupdf pypdf sentence_transformers rich wget python-dotenv cryptography unstructured accelerate

NOTE: you may receive an error message such as `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.`. This is a known issue and can be ignored.

#### 1.1.1 Setting up the environment (Kaggle or Google Colab)

This update package list and install curl, which then use to fetch Ollama installation script.

In [ ]:
# Update system files and install curl
!apt-get update && apt-get install -y curl

#### 1.1.2 Installing Ollama (Kaggle or Google Colab)

This will install Ollama automatically.

In [ ]:
# Install Ollama
!curl https://ollama.ai/install.sh | sh

#### 1.1.3 Starting the Ollama server (Kaggle or Google Colab)

Here, we use Python’s `subprocess.Popen()` to run the Ollama server in the background, listening on the specified host and port (127.0.0.1:11438). The server will be available for communication via the API.

You will need this address later for the RAG model.

In [ ]:
import subprocess
import time
import os

# Set the environment variable for the Ollama host and port
os.environ['OLLAMA_HOST'] = '127.0.0.1:11438'

# Start the Ollama server in a subprocess
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to start
print("Starting Ollama server...")
time.sleep(1)  # Wait for the server to initialize

print("Ollama server is running on 127.0.0.1:11438")

#### 1.1.4 Pulling an open-source LLM model (Kaggle or Google Colab)
This command downloads the llama3.2:1b model from Ollama’s servers (a small version of LLAMA 3.2), making it available for use on your notebook machine environment.
Also, refer here https://ollama.com/search for checking the list of models available on Ollama model repository.

In [ ]:
# Pull a model (e.g., llama)
model_name = 'llama3.2:1b'
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', model_name])

#### 1.1.4 Interacting directly with the model (Kaggle or Google Colab)

To test the installation and the model, we can use the following code block to interact with the model via HTTP requests.

In [ ]:
import requests
import json

def generate_response(query):
    # Define the API URL for the same host as mentioned when starting the server.
    url = "http://localhost:11438/api/chat"
    
    # Define the payload (data) to send in the POST request
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ],
        "keep_alive": 0,
        "stream": False
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    # Make the POST request to the API
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            output = response.text
            return output
        else:
            print(f"Failed to get a response. Status code: {response.status_code}")
            return ("Error response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Once the server is running and the model is pulled, you can start interacting with the model using Python code. In this example, the `generate_response()` function sends a POST request to the Ollama API, passing in the user query. The model processes the query and returns a generated response.

*Note: If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull. Also, check the model name to match with the model you pulled using `ollama pull` command.*

In [ ]:
# Using the function to get the output
# The output is in JSON format, and you can parse it using Python’s json module.

query = "What is the difference between AI and ML? Give a short answer."
output = generate_response(query)
json.loads(output)['message']['content']

#### 1.1.5 Using Ollama’s Python SDK (Kaggle or Google Colab)

To interact with the Ollama server, you can use the Python SDK. The SDK is available on PyPI and can be installed using pip. The Ollama python SDK is a wrapper around the Ollama API, which allows you to interact with the server using Python code. The SDK provides a simple and easy-to-use interface for sending requests to the server and receiving responses from the model. This is what we will use in this lab to interact with the model.

In [ ]:
!pip install ollama

The code below interacts with the Ollama model using the Python SDK to send a query and print the generated response.

*Note: (again) If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull.*

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': 'Show me the lyrics of any song?',
  },
])
print(response['message']['content'])

If all works, you should see the generated response from the model. In my test, I got the lyrics from the song "Yesterday" by The Beatles. What did you get?

- - - 

### 1.2 Installing Ollama (locally on your machine)

This guide will help you to install Ollama on your local machine. You need a fairly good machine to run LLMs models and space in your hard disk to store the LLM models (several GB).  If you have a machine with a GPU, you can run the models faster.
However, there are lighter models that can run on CPU and provide results good enough for this lab and many applications (e.g., you can try "llama3.2:1b", or "qwen2.5:0.5b" model).

#### 1.2.1 Download and install Ollama
Ollma is available for macOS, Linux, and Windows. Go to the official [website](https://ollama.com/) and download the version for your operating system. Follow the instructions to install it.

#### 1.2.2 Running Ollama
After installing Ollama, you should see the Ollama icon in your system tray. You can use it to see the logs and quit the application.

Usuall, after the installation Ollama is *already* running. 

1. [Optional] If you want to run it again, you can run the following command in your terminal:

    ```bash
    ollama serve
    ```

2. To pull a model, go to the [Ollama model list](https://ollama.com/search) and select a model suitable for your machine. Start humble with a small model. You can for instance try the small version of [LLAMA 3.2](https://ollama.com/library/llama3.2).
Follow the instractions for the model you selected. For example, we suggest you try LLAMA 3.2 (the 1B parameters version). To use this version, run in your terminal:

    ```
    ollama run llama3.2:1b
    ```
    The `run` command will pull (download) the model to your machine and then run it, exposing it via the API started with `ollama serve`.

3. [Optional] If you want to *pull* a differen model, you can run:

    ```bash
    ollama pull <model_name>
    ```
    The commanda `ollama list` will show you the list of models available on your machine. To switch between models available on your machine, you can just use the `run` command. The `stop` command will stop a running model.

NOTES:
- If the model is not present in your machine, the `run` command will autamtically try to pull it
- Models takes a lot of space in your hard disk. Make sure you have enough space before pulling a model. Remove unnecessary models using the `ollama remove <model_name>` command.

#### 1.2.3 Get Ollama address
In order to interact with the model, you need to know the address where the model is running. Ollama binds to the local address 127.0.0.1 on port 11434 by default.

This means that you can note http://localhost:11434

#### 1.2.4 Package installation (with Poetry)
We will install all the necessary packages (ollama-related packages included) using Poetry. We provide a `pyproject.toml` file with the necessary dependencies. We assume you have [Poetry installed](https://github.com/hei-synd-aml/lab-0-TutoPoetry) on your machine. Open a terminal in the project location and run the following command to install the packages.

```bash
poetry install
```

---

## 2. Creating a prompt and interacting with the model

OK, now that you have the Ollama server running (either on Kaggle or on your Local machine) and you pulled the model, let's start by creating a prompt and interacting with the LLM model.
In this section, we will create a prompt and interact with the model to generate text.

### Imports

In [1]:
import os
from pathlib import Path

import wget
from dotenv import load_dotenv
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from rich.console import Console
from rich.markdown import Markdown

console = Console()

### Constants

In [2]:
load_dotenv(override=True)

OLLAMA_ADDRESS = "http://localhost:11434"  # replace with your OLLAMA address, usually http://localhost:11434
LLM_NAME = "llama3.2:1b"  # replace with the LLM name you pulled from the OLLAMA


### Connecting to LLM

NOTE: if you get "ConnectionError", check if the Ollama server is running and the address is correct.

In [3]:
llm = OllamaLLM(
    model=LLM_NAME,
    base_url=OLLAMA_ADDRESS,
    temperature=0.1,  # Will be explained later
    stop=["<end_of_turn>"],
)

llm.generate(["Hello, how are you today?"]).generations[0][0].text

"I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help you with any questions or topics you'd like to discuss. How about you? How's your day going so far?"

**Question**:
- What is the role of the temperature parameter in a LLM?

**Answer**:

- TO DO

### Creating a prompt

A prompt is generally divided into two parts: the context and the question.

The context provides the information that the model will use to generate its answer, while the question specifies what the model is expected to respond to.
Typically, the context contains a system prompt, which explains the expected behavior from the model, and other additional information that can help the model to provide the right answer. Typically, when used for chatbots, in the context there is a summary (or the entierity!) of all the previous exchange the user had with the model in that conversation. 

LangChain uses templates and markers indicating where to insert the user's question and the context within the template. For more infomation, you can check [Langchain prompt templates documentation](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates).

In [4]:
# A langchain's template is nothing more than a string with {markers} indicating placeholder where other text will be inserted when creating a prompt.

template = """
You are an helpful assistant that answer the question in detail.

Human input: {question}
Assistant:"""

prompt = PromptTemplate(input_variables=["question"], template=template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpful assistant that answer the question in detail.\n\nHuman input: {question}\nAssistant:')

### Creating the chain and start a conversation

In [5]:
# Simple chain putting together a prompt and a model 
simple_llm_chain = prompt | llm

In [6]:
result = simple_llm_chain.invoke(input="When is held the conference called AI-days 2025?")

console.print(Markdown(result)) # we print the output proposed by the LLM

I can provide you with information on the AI Days 2025 conference. However, I need to clarify a few things first.  

After conducting research, I found that there are several conferences and events related to Artificial Intelligence
(AI) that take place around the world. Some of these events might be similar or even identical to the "AI Days" you
mentioned.                                                                                                         

To provide more accurate information, could you please provide me with some additional context or details about the
AI Days 2025 conference? For example:                                                                              

 • What is the location of the conference?                                                                         
 • Who is the organizer or sponsor of the conference?                                                              
 • What topics or themes will be covered during the conference?                                                    
 • Are there any specific speakers or researchers that have been announced?                                        

Once I have more information, I can try to provide you with a more accurate answer about when and where the AI Days
2025 conference takes place.

**Questions**:
- What is the relationship between the `input` parameter in `llm_chain.invoke(input="")` and the prompte template defined above?
- Did the model generate a good answer? Why?
- Feel free to change the prompt and the input and see what happens. Try to consider this point: how can you judge the quality of the answer? How could you create a proper evaluation protocol to evaluate the quality of the answers? How is it done in the industry/research field?

**Answers**:
- TO DO




---

## 3. Using the RAG

For the moment, we have a model that can generate answers to questions. To provide answers, it only uses its knolwege got at training time and the context we provide. But what if we could provide the model with a set of documents and ask it to retrieve the most relevant ones to answer the question?
That is the idea behind the Retrieval-Augmented Generation (i.e., literally, we augment the generation of text by enriching the context with information contained in relevant documents).

A typical use case is a company having a huge internal knowledge base (e.g., a collection of documents, web pages, etc.) and wanting to provide its employees with a chatbot that can answer questions about the knowledge base. The chatbot will retrieve the most relevant documents from the knowledge base and use them to generate the answer. This is particular interesting for companies that have a lot of internal documents that may contain sensitve data.

<img src="./img/RAG_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://www.promptingguide.ai/research/rag)

### Downloading the pdfs

Here we demonstrate RAG using a collection of *pdf* documents. We will use the [Swiss AI Center](https://www.hes-so.ch/swiss-ai-center) website as a source of documents. The documents are available in the `data` folder. You are free to test the code with your own documents. Just make sure they are in the `data` folder.

NOTE: *pdf* documents are surprisingly complex documents. They can contain text, images, tables, etc. Actually, two different *pdf* documents can have very different structures. Some additional preprocessing may be needed to extract the text from the pdfs. However, for the purpose of this lab, the code provided below should work for most of the documents.

In [7]:
# Create the "data/PDFs" folder if it doesn't exist
PDF_FOLDER = Path("data/PDFs")
os.makedirs(PDF_FOLDER, exist_ok=True)

urls = [
    "https://ai-days.swiss-ai-center.ch/PDF/Session1.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2b.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3b.pdf",
]

# Download the PDFs
for url in urls:
    name = url.split("/")[-1]
    if not (PDF_FOLDER / name).is_file():
        filename = wget.download(url, f"data/PDFs/{name}")
console.print("Pdf file downloaded successfully.", style="bold green")

Pdf file downloaded successfully.

### Loading and preprocessing PDF files

In [8]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '(?<=\. )', '(?<=\, )', ' ', '']
)

***Questions***
- Investigate and explain the role of `chunk_size` and `chunk_overlap` in the code above.
- What are the advantages and disadvantages of having bigger/smaller `chunk_size`?
- What are the advantages and disadvantages of having bigger/smaller `chunk_overlap`?
- What is a difference between a token and a chunk?
- What is the parameter `separators` used for?
- What does the **Recursive** Character Text Splitter does differently compared to a **Simple** character text splitter? 

***Answers***
- TO DO

In [9]:
# All data will be in the list documents
documents=[]

In [10]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
# Load and process the text files
# loader = TextLoader('single_text_file.txt')
loader = DirectoryLoader(PDF_FOLDER, glob="./*.pdf", loader_cls=PyPDFLoader)

pdf_docs = loader.load()
len(pdf_docs)

# tokenize pdf
documents.extend(text_splitter.split_documents(pdf_docs))
len(documents)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong po

44

#### What is a "Document" in langchain?
In LangChain, a  [document](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html) is composed of two elements: the text (contained in the field `page_content`) and the metadata (contained in the field `metadata`).

The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [11]:
# To understand what is going on, we print a document. The text:
print(documents[1].page_content)

Session 1: Compute Infrastructures for IA applications in the wild Location: Movie theater 6 With the advent of Chatbots, LLMs and other generative IA technologies, as well as other progresses in the IA ﬁeld, there is an explosion of the demand for compute force. IA is no longer computer science: it is computational science. As such, it can no longer be done with casual, self-managed equipment. More advanced compute infrastructures are required both to satisfy user needs (in terms of compute


In [12]:
# To understand what is going on, we print a document. The text:
print(documents[1].metadata)

{'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20250116093703Z00'00'", 'title': 'Workshop_descriptions', 'moddate': "D:20250116093703Z00'00'", 'source': 'data\\PDFs\\Session1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


**Question**:
- why is important to also store metadata when doing RAG? Any idea?

**Answer**:
- TO DO

### Embedding a PDF in a vectorstore

The code below creates a vectorstore from the documents. The vectorstore is a database that stores the embeddings of the documents. The embeddings are used to retrieve the most relevant documents for a given question.
It may take a while to create the vectorstore, depending on the number of documents and the size of the documents.

However, if you have new documents, you can just add them to the vectorstore using the `add_documents` method instead of recreating the vectorstore from scratch. 

In [13]:
VECTORSTORES_DIR = Path("data/vectorstores_pdf")

In [14]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5" # locally you can also use smaller models. Check the HuggingFace model hub for more models with different sizes, performance and languages: https://huggingface.co/spaces/mteb/leaderboard

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    # to use the "cuda" configuration, you need an nvidia GPU, and to install 
    # On Kaggle, you set to use as accelerator GPU P100 (you need a verified account)
    # model_kwargs={"device": "cpu"}, # change to cuda -> cpu if you do not have a Nvdia GPU
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

The instructions below embeds the documents in the vectorstore. Then, we can save the vectorstore to disk. In this way, we can load the vectorstore and the documents at a later time without having to re-embed the documents. If we get new relevant documents, we can add them to the vectorstore and re-embed them. If interested, you can check the [FAISS documentation](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html) for more information.

In [15]:
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

In [16]:
vectorstore.save_local(VECTORSTORES_DIR)

### Loading a vectorstore
We already have the vectorstore with the pdfs embedded in the variabale `vectorstore`. For completeness, in the code above, we show how to load the vectorstore from disk.

In [17]:
vectorstore = FAISS.load_local(
    VECTORSTORES_DIR, embedding_model, allow_dangerous_deserialization=True
)

### New prompt

In RAG we need to add another marker to indicate where the new information (or context) should be inserted for this we use the variable named `{context}`.

In [18]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Use three sentences maximum and keep the answer as concise as possible.
You must answer in english. If you don't know the answer, say "I don't know".
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nDon\'t try to make up an answer and only use the information you know.\nUse three sentences maximum and keep the answer as concise as possible.\nYou must answer in english. If you don\'t know the answer, say "I don\'t know".\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n')

***Questions***
- If you look closely, you will notice that the prompt above has 4 regions into which we can insert (or not) information (the system prompt, the context, the input and the answer). What are these regions? Explain the role of each one.

***Answers***
- TO DO

**Tips and Tricks**: You can modify the system prompt to get an answer closer to the one you are looking for.

- Change the language, style, or length.
- Add more information to the prompt, such as suggesting a style or tone for the answer (e.g., "be concise", "be formal", "be informal", "be friendly", etc.).
- Provide more context (e.g., "the text is a scientific paper", "the text is a news article", etc.).
- Specify the question type (e.g., "the question is about the content of the text", "the question is about the author of the text", etc.).
- Define the answer format (e.g., "the answer should be a summary of the text", "the answer should be a list of keywords", etc.).
- Include information about the user (e.g., "the user is a scientist", "the user is a student", etc.).
- Mention details about the model (e.g., "the model is a chatbot", "the model is a search engine", etc.).
- Indicate the desired output format (e.g., "the output should be in JSON format", "the output should be in XML format", etc.).
- Provide specific examples to illustrate the impact of prompt modifications.
- Highlight any limitations or constraints of the model.
- And more...

**Notes and Warnings**:

- Different models may have different markers for the question and context. Check the documentation of the model you are using. This is a true challenge of prompt-engineering.
- Some models (especially smaller ones) may not support different languages.

### Similarity search

<img src=".\img\Similarity_search_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://python.langchain.com/docs/concepts/vectorstores/)

In [19]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In the code above, multiple things go on at the same time.
[create_stuff_documents_chain](https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/) combines documents by concatenating them into a single context window. 

**Questions**:
- What is a retriever? 
- What is the meaning of the parameter: "mmr"? What is the difference between "mmr", "similarity" and "similarity_score_threshold"?
- What is the meaning of the parameter "k" and how increasing/decreasing it may impact the results?
- When using "mmr", there are other parameters for the retriever. One of them is `lambda_mult`. What is the meaning of this parameter and what are the possible values? 
- What `create_retrieval_chain` does?

**Answers**:
- TO DO

### "Chatting" with a pdf

Everything is ready to chat with the model: the content of the pdf is embedded in the vectorstore, the model is running, and the chain is created. Let's ask some question to our pdf! Maybe they can help, where the model alone could not.

In [20]:
result = chain_with_retriever.invoke(input={"input": "When is held the conference called AI-days 2025?"})

console.print(Markdown(result["answer"]))

The conference is called AI-DAYS@HES-SO 2025, which means it is being held in January 2025 at HES-SO (Swiss        
Institute of Applied Sciences) in Geneva and Lausanne.

**Question**:
- What is the size of `result`? What do you expect to find in `result`?
- Let us consider again the "temperature" parameter. For an application that needs to provide accurate information coming from existing document, is it better to use a high or low temperature?
- Suggestion: try to change it and see the difference in the answer.

**Answer**:
- TO DO

### Explaining the output

The `result` dictionary contains much more than the simple answer. Thanks to the metadata, we can see the title of the document, the page number, and the context where the answer was found!

In [21]:
console.print(result)

{
    'input': 'When is held the conference called AI-days 2025?',
    'context': [
        Document(
            id='c2986068-dc35-40e0-b74f-fe6553316610',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='58326228-0bdb-474d-be87-d076224d0677',
            metadata={
                'producer': 'macOS Version 15.0.1 (Build 24A348) Quartz PDFContext, AppendMode 1.1',
                'creator': 'Word',
                'creationdate': "D:20241105080725Z00'00'",
                'moddate': "D:20241105080730Z00'00'",
                'title': 'Workshop_descriptions',
                'source': 'data\\PDFs\\Session2a.pdf',
                'total_pages': 2,
                'page': 0,
                'page_label': '1'
            },
            page_content='Session 2a: Edge AI Tools, devices, and methods With the advent of Chatbots, LLMs and 
other generative IA technologies, as well as other progresses in the IA ﬁeld, there is an explosion of the demand 
for compute force. IA is no longer computer science; it is computational science. As such, it can no longer be done
with casual, self-managed equipment. More advanced compute infrastructures are required both to satisfy user needs 
(in terms of compute power, GPU Ram capacity) and to ensure a decent'
        ),
        Document(
            id='cc4ec967-3948-4d08-8da8-ca6f28e45f01',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content="Schedule: 8h30-12h20 + 13h15-17h15 8:30:00 AM 0:05 Opening remarks Sébatien Rumley 
HEIA-FR, HES-SO / Swiss Ai center 8:35:00 AM 0:30 The Alps research infrastructure at CSCS: enabling world-class ML
research in Switzerland Fawzi Mohamed The Swiss National Supercomputing Centre (CSCS), ETH Zurich 9:05:00 AM 0:18 
SCITAS: On-premise and Cloud Infrastructure driving HPC & AI Scientific Computing at EPFL Gilles Fourestey 
SCITAS/EPFL 9:23:00 AM 0:18 Picterra's Infrastructure: Scaling ML for"
        ),
        Document(
            id='97effe50-c711-4ab4-a5ec-d9ad79d53012',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content="Session 2b:  AI for Local Energy Systems  Ce workshop est organisé par le projet phare de
la HES-SO : Smart Energy District. Il vise à présenter quelques projets dans le domaine de l’AI appliquée au 
secteur de l’énergie, en particulier les « local energy communities ». Il s’adresse aux Gestionnaires de Réseaux de
Distribution (GRDs), aux communes et municipalités ainsi qu’aux entreprises qui opèrent dans ce domaine.  This 
workshop is organised by the HES-SO's ﬂagship project: Smart Energy"
        ),
     

**Questions**:
- The `result`dictionary contains thee keys: `input`, `context`, and `answer`. What do they contain? Do they remember you of something?
- Let us focus on the `context` key. It contains a list of documents. Each document is a dictionary with three keys: `id`, `metadata`, and `page_content`. What do they contain? What is the relationship between what you see and the "chunk" you saw before?
- Which of this information is used by the retriever to retrieve the documents?
- Which of this information is used by the LLM to generate the answer?

**Answers**:
- TO DO

### Add filters using metadata
In the code above, we did not yet used information in metadata. Here below, weshow how to filter the documents based on just that. For instance, what if we want to limit (filter) the retrieval to document with a specific title?

In [25]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
        
        # this below is the only new part
        "filter": { 
            "title": "session2b"  # Filter by title
        }
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [26]:
result = chain_with_retriever.invoke(input={"input": "When is held the conference called AI-days 2025?"})

console.print(Markdown(result["answer"]))

I don't know.

In [27]:
console.print(result)

{
    'input': 'When is held the conference called AI-days 2025?',
    'context': [
        Document(
            id='9f2f5a80-aa63-4949-961e-9e632fa65490',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='97effe50-c711-4ab4-a5ec-d9ad79d53012',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content="Session 2b:  AI for Local Energy Systems  Ce workshop est organisé par le projet phare de
la HES-SO : Smart Energy District. Il vise à présenter quelques projets dans le domaine de l’AI appliquée au 
secteur de l’énergie, en particulier les « local energy communities ». Il s’adresse aux Gestionnaires de Réseaux de
Distribution (GRDs), aux communes et municipalités ainsi qu’aux entreprises qui opèrent dans ce domaine.  This 
workshop is organised by the HES-SO's ﬂagship project: Smart Energy"
        ),
        Document(
            id='3a071aa4-b889-4e0e-9407-3ea204ee7857',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content='15h55 – 16h05 Swiss Energypark – laboratoire des ﬂexibilités en conditions réelles 
Laurent Raeber, Swiss Energy Park'
        ),
        Document(
            id='f5fdd733-1310-4e8c-9c9f-ae30ca05f399',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content='16h05 – 16h15 Le projet AISOP Antonios Papaemmanouil, HSLU 16h15 – 16h30 Pause-Café  
16h30 – 17h30 Table ronde. Animateur : Rafael Tiedra (OCSIN, canton de Genève) 1. Dario Poroli (Commune de Meyrin) 
2. SIG (TBC) 3. Mokhtar Bozorg, HES-SO 4. Olivier Crettenand, SD Energy 5. Laurent Raeber, Swiss Energy Park 6. Max
Carrel, Dynao  Notes The workshop has no speciﬁc registration, and walk-ins are welcome. Workshop committee Last 
Name First Name Institution  e-mail address Abdennadher  Nabil HES-SO'
        ),
        Document(
            id='30657fe1-1a76-4685-9a29-f62ffdd85466',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
               

**Questions**:
- Was the answer generated by the model different? 
- Did you notice that the answer was more or less more precise? Why?

**Answers**:
- TO DO

---

## 4. Challenge: getting information from a website

Now, it's your turn. Use the code above as inspiration to:
- interact with a website (instead of a PDF)
- create a vectorstore with the information from the website
- save the vectorstore to disk
- add embeddings from another website to the vectorstore (without recreating it from scratch)
- improve the prompt template to get better answers
- chat with the model
- evaluate the output (with and without RAG)

You can choose any website you want. If do not have any inspiration, we suggest to use https://docs.realgames.co/homeio/en/.
You may ask the model to provide you with a list the available detectors in Home I/O. And try to use filters and metadata to improve the results (maybe pointing the actual webpage containing the correct answer...).


Hint: If you need some help to get started check [this](https://python.langchain.com/docs/how_to/document_loader_web/).

In [ ]:
# TODO:

## Conclusion

In this lab, we learned how to create a simple RAG system to answer questions about pdf documents and websites. We used a vector database to store information about the pdf documents and a self-hosted LLM to generate the answers. We also learned how to create prompts and interact with the model to generate text.

Pdf and web documents are just examples of the many applications of RAG systems. RAG can work with any type of document or data source, such as databases, APIs, YouTube videos, excel files, etc. 